# Gaussian Drift OPT Debug

Minimal notebook for testing only the homogeneous Gaussian initial-condition drift. FD is the control; OPT is the object under test. Shared helpers live in `temporaryHelpers.jl`.

## 1. Setup

In [ ]:
using Pkg
cd(@__DIR__)
Pkg.activate("../")

try
    using Metal
catch err
    @warn "Metal not loaded; continuing on CPU" err
end
ParamFile = "../config/testparam.csv"

include("../src/batchFiles/batchGPU.jl")
include("../src/commonBatchs.jl")
include("../src/planet1D.jl")
include("../src/GeoPoints.jl")
using .commonBatchs, .planet1D, .GeoPoints

include("../src/flexOPT.jl")
using .flexOPT

include("temporaryHelpers.jl")


## 2. Controls And Initial Gaussian

In [ ]:
# Homogeneous toy controls
shape = (201, 201)
velocity_value = 2600.0
dx = 100.0
cfl = 0.45
Nt = 180
store_every = 3
sigma = 10.0
amplitude = 1.0

# OPT controls
pointsInSpace = 3
pointsInTime = 3
supplementaryOrder = 2
orderBspace = 1
orderBtime = 1

center = CartesianIndex(cld(shape[1], 2), cld(shape[2], 2))
dt = cfl * dx / velocity_value
delta = (dx, dx, dt)
velocity = fill(velocity_value, shape)
init_gaussian = gaussian_field(shape, center; sigma=sigma, amplitude=amplitude)

@show shape center velocity_value delta sigma Nt store_every


## 3. FD Baseline

In [ ]:
preparedFD = prepare_fd2d_acoustic_baseline(velocity, delta)
frames_fd = propagate_linear_frames_from_initial(
    preparedFD,
    init_gaussian,
    init_gaussian,
    Nt;
    store_every=store_every,
    blowup_limit=1e12,
)

fd_report = wavefield_snapshot_report(frames_fd)
fd_drift = drift_report(frames_fd, center)
fd_argmax = argmax_report(frames_fd, center)
fd_symmetry = symmetry_report(frames_fd, center)

@show length(frames_fd) fd_report[1] fd_report[end]
@show fd_drift[1] fd_drift[end]
@show fd_argmax[1] fd_argmax[end]
@show fd_symmetry[1] fd_symmetry[end]


In [ ]:
fig_fd = plot_wave_snapshots(
    frames_fd;
    sourcePoint=center,
    title="FD baseline: Gaussian initial condition",
)
fig_fd


## 4. OPT Build

In [ ]:
toyOPT = build_toy_opt_prepared(
    velocity_value=velocity_value,
    shape=shape,
    dx=dx,
    cfl=cfl,
    pointsInSpace=pointsInSpace,
    pointsInTime=pointsInTime,
    supplementaryOrder=supplementaryOrder,
    orderBspace=orderBspace,
    orderBtime=orderBtime,
)
preparedOPT = toyOPT.prepared

@show preparedOPT.spaceShape preparedOPT.NpointsSpace preparedOPT.NField preparedOPT.NForceField preparedOPT.timePointsUsedForOneStep
@show size(preparedOPT.A_unknown) nnz(preparedOPT.A_unknown) nnz(preparedOPT.L_known)

opt_A_report = implicit_matrix_report(preparedOPT)
@show opt_A_report


In [ ]:
toyOPT.optRec["recette"].lhs.Ajiννᶜ

In [ ]:
audit = recipe_local_index_audit(toyOPT.optRec)
audit

In [ ]:
center = CartesianIndex(cld(size(toyOPT.velocity,1),2), cld(size(toyOPT.velocity,2),2))

st = operator_stencil_at_point(toyOPT.numOps, center; which=:left, iExpr=1, iField=1)
stencil_time_summary(st)
st

## 5. OPT Propagation

In [ ]:
frames_opt = propagate_linear_frames_from_initial(
    preparedOPT,
    init_gaussian,
    init_gaussian,
    Nt;
    store_every=store_every,
    blowup_limit=1e12,
)

opt_report = wavefield_snapshot_report(frames_opt)
opt_drift = drift_report(frames_opt, center)
opt_argmax = argmax_report(frames_opt, center)
opt_symmetry = symmetry_report(frames_opt, center)

@show length(frames_opt) opt_report[1] opt_report[end]
@show opt_drift[1] opt_drift[end]
@show opt_argmax[1] opt_argmax[end]
@show opt_symmetry[1] opt_symmetry[end]


In [ ]:
fig_opt = plot_wave_snapshots(
    frames_opt;
    sourcePoint=center,
    title="OPT: Gaussian initial condition",
)
fig_opt


In [ ]:
Main.flexOPT.opt_integral_order = :ln_lc
toy_a = build_toy_opt_prepared(velocity_value=velocity_value,
    shape=shape,
    dx=dx,
    cfl=cfl,
    pointsInSpace=pointsInSpace,
    pointsInTime=pointsInTime,
    supplementaryOrder=supplementaryOrder,
    orderBspace=orderBspace,
    orderBtime=orderBtime,)
st_a = operator_stencil_at_point(toy_a.numOps, center; which=:left)
stencil_time_summary(st_a)
st_a

In [ ]:
Main.flexOPT.opt_integral_order = :lc_ln
toy_b = build_toy_opt_prepared(velocity_value=velocity_value,
    shape=shape,
    dx=dx,
    cfl=cfl,
    pointsInSpace=pointsInSpace,
    pointsInTime=pointsInTime,
    supplementaryOrder=supplementaryOrder,
    orderBspace=orderBspace,
    orderBtime=orderBtime,)
st_b = operator_stencil_at_point(toy_b.numOps, center; which=:left)
stencil_time_summary(st_b)
st_b

## 6. FD vs OPT Drift Comparison

In [ ]:
drift_comparison = (
    fd_final_drift = fd_drift[end],
    opt_final_drift = opt_drift[end],
    fd_final_argmax = fd_argmax[end],
    opt_final_argmax = opt_argmax[end],
    fd_final_symmetry = fd_symmetry[end],
    opt_final_symmetry = opt_symmetry[end],
)
drift_comparison


## 7. Optional OPT Parameter Sweep

In [ ]:
# Optional: change these values and rerun to see whether drift is caused by OPT parameters.
# Some implicit OPT future blocks can be singular or non-finite; those cases are reported and skipped.
parameter_cases = [(3,2), (3,0), (5,2)]

sweep_rows = NamedTuple[]
for (pspace, supp) in parameter_cases
    println("building OPT case pointsInSpace=$pspace supplementaryOrder=$supp")
    a_report = nothing
    try
        toy = build_toy_opt_prepared(
            velocity_value=velocity_value,
            shape=shape,
            dx=dx,
            cfl=cfl,
            pointsInSpace=pspace,
            pointsInTime=pointsInTime,
            supplementaryOrder=supp,
            orderBspace=orderBspace,
            orderBtime=orderBtime,
        )
        a_report = implicit_matrix_report(toy.prepared)
        if a_report.stored_nonfinite > 0 || a_report.zero_rows_finite > 0
            push!(sweep_rows, (
                pointsInSpace=pspace,
                supplementaryOrder=supp,
                status=:bad_A_unknown,
                matrix_report=a_report,
                nframes=0,
                final_drift=nothing,
                final_symmetry=nothing,
                final_max=NaN,
            ))
            @warn "Skipping OPT case because A_unknown is non-finite or has finite-zero rows" pointsInSpace=pspace supplementaryOrder=supp matrix_report=a_report
            continue
        end
        fr = propagate_linear_frames_from_initial(toy.prepared, init_gaussian, init_gaussian, Nt; store_every=store_every, blowup_limit=1e12)
        dr = drift_report(fr, center)
        sy = symmetry_report(fr, center)
        push!(sweep_rows, (
            pointsInSpace=pspace,
            supplementaryOrder=supp,
            status=:ok,
            matrix_report=a_report,
            nframes=length(fr),
            final_drift=dr[end],
            final_symmetry=sy[end],
            final_max=maximum(abs, fr[end]),
        ))
    catch err
        push!(sweep_rows, (
            pointsInSpace=pspace,
            supplementaryOrder=supp,
            status=Symbol(nameof(typeof(err))),
            matrix_report=a_report,
            nframes=0,
            final_drift=nothing,
            final_symmetry=nothing,
            final_max=NaN,
        ))
        @warn "Skipping failed OPT case" pointsInSpace=pspace supplementaryOrder=supp matrix_report=a_report exception=(err, catch_backtrace())
    end
end
sweep_rows
